In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import plotly.graph_objects as go
import plotly.io as pio
import json
from datetime import datetime

pio.renderers.default = 'vscode'
pio.templates.default = 'plotly_white'

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
train = np.load('/home/arseniy/Document/Unic/Program/Прочее/Plat_ml/data/processed/test.npz')
val   = np.load('/home/arseniy/Document/Unic/Program/Прочее/Plat_ml/data/processed/val.npz')
test  = np.load('/home/arseniy/Document/Unic/Program/Прочее/Plat_ml/data/processed/test.npz')

def to_loader(X, y, batch_size=32, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32)
    ds = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(train['X'], train['y'], batch_size=32, shuffle=True)
val_loader   = to_loader(val['X'],   val['y'],   batch_size=32, shuffle=False)
test_loader  = to_loader(test['X'],  test['y'],  batch_size=32, shuffle=False)

print(f"Батчей: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

Батчей: train=4, val=4, test=4


In [3]:
class Pt100Net(nn.Module):
    def __init__(self, hidden_sizes=(64, 64, 32), dropout=0.1):
        super().__init__()
        layers = []
        prev = 1
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.Tanh(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = Pt100Net().to(device)
print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nОбучаемых параметров: {n_params}")

Pt100Net(
  (net): Sequential(
    (0): Linear(in_features=1, out_features=64, bias=True)
    (1): Tanh()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): Tanh()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): Tanh()
    (8): Dropout(p=0.1, inplace=False)
    (9): Linear(in_features=32, out_features=1, bias=True)
  )
)

Обучаемых параметров: 6401


In [4]:
class Pt100Net(nn.Module):
    def __init__(self, hidden_sizes=(64, 64, 32), dropout=0.1):
        super().__init__()
        layers = []
        prev = 1
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.Tanh(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = Pt100Net().to(device)
print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nОбучаемых параметров: {n_params}")

Pt100Net(
  (net): Sequential(
    (0): Linear(in_features=1, out_features=64, bias=True)
    (1): Tanh()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): Tanh()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): Tanh()
    (8): Dropout(p=0.1, inplace=False)
    (9): Linear(in_features=32, out_features=1, bias=True)
  )
)

Обучаемых параметров: 6401


In [7]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=20
)

In [5]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = criterion(pred, yb)
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

In [8]:
EPOCHS = 1000
PATIENCE = 50
best_val = float('inf')
best_state = None
no_improve = 0
history = {'train': [], 'val': []}

for epoch in range(EPOCHS):
    tr_loss = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss = evaluate(model, val_loader, criterion)
    history['train'].append(tr_loss)
    history['val'].append(vl_loss)
    scheduler.step(vl_loss)

    if vl_loss < best_val:
        best_val = vl_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping на эпохе {epoch+1}")
            break

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:4d} | train={tr_loss:.6e} | val={vl_loss:.6e}")

model.load_state_dict(best_state)
print(f"\nЛучший val loss: {best_val:.6e}")

Epoch   50 | train=5.077463e-03 | val=2.253938e-04
Epoch  100 | train=4.478035e-03 | val=1.997260e-04
Epoch  150 | train=3.964375e-03 | val=1.218406e-04
Epoch  200 | train=3.209223e-03 | val=1.364198e-04
Epoch  250 | train=3.625131e-03 | val=1.629185e-04
Epoch  300 | train=4.000473e-03 | val=1.237967e-04
Epoch  350 | train=2.682065e-03 | val=1.154869e-04
Early stopping на эпохе 360

Лучший val loss: 1.032697e-04


In [9]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=history['train'], mode='lines', name='train',
    line=dict(color='steelblue')
))
fig.add_trace(go.Scatter(
    y=history['val'], mode='lines', name='val',
    line=dict(color='orange')
))

fig.update_yaxes(type='log', title_text='MSE (normalized)')
fig.update_xaxes(title_text='Epoch')
fig.update_layout(
    height=500,
    title_text='Кривые обучения',
    title_x=0.5,
    hovermode='x unified'
)
fig.show()

In [10]:
import os
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

torch.save({
    'model_state': model.state_dict(),
    'hidden_sizes': (64, 64, 32),
    'dropout': 0.1,
    'best_val': best_val,
    'epochs_trained': len(history['train']),
}, 'models/pt100_model.pth')

metrics = {
    'timestamp': datetime.now().isoformat(),
    'model': 'Pt100Net',
    'hidden_sizes': [64, 64, 32],
    'dropout': 0.1,
    'epochs_trained': len(history['train']),
    'best_val_loss': float(best_val),
}
with open('results/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Модель и метрики сохранены.")

Модель и метрики сохранены.
